# GCM BA - 1. Check of the inputs (nothing is written)

Notebooks 2 and 3 read their inputs **directly** from the input files (no copies), through `ba_utils.gcm_ba_input`,
which standardises them when reading (`read_raw`: degC, mm/day, `noleap` calendar, days at 00:00, `pr` of the GCM runs
glued from its NH + SH files):
- **ERA5** regridded **conservatively** onto the EC-Earth3 grid (Gaussian N128, 256 x 512) **with CDO** on JEO-desk
  (`GCM_downscaling/Input_data/ERA5_daily_on_ecearth3grid/`, dataset `era5_on_ecearth3grid` of notebook 00);
- **EC-Earth3 historical**, merged with CDO (`GCM_downscaling/Input_data/EC-Earth3_historical_CMIP6_merged/`);
- the **four GCM runs** (2025 / 2050, original / hosing) in `EC-Earth_hosing/`, already on the EC-Earth3 grid.

F1 is computed on the **whole globe**, without land mask (F1 works pixel by pixel). Variables: pr, tasmax, tasmin
(`rsds` is not bias-adjusted).

This notebook checks, before the calibration: every input on exactly the EC-Earth3 grid, periods and number of days,
and the CDO regridding against the Python one (`ba_utils.regrid_conservative`).

CDO commands used (JEO-desk, environment `~/envs/cdo`), in `GCM_downscaling/Input_data/`:
```bash
cdo griddes EC-Earth3_historical_CMIP6_merged/tasmax_day_EC-Earth3_historical_r1i1p1f1_gr_19810101-20141231.nc > ~/ecearth3_grid.txt
cdo gencon,$HOME/ecearth3_grid.txt ERA5_daily_merged/pr_ERA5_day_1981-2020.nc ~/weights_era5_to_ecearth3.nc
cdo -P 8 -z zip_1 remap,$HOME/ecearth3_grid.txt,$HOME/weights_era5_to_ecearth3.nc \
    ERA5_daily_merged/pr_ERA5_day_1981-2020.nc ERA5_daily_on_ecearth3grid/pr_ERA5_day_1981-2020.nc
# same for tasmax and tasmin
```

```
elisa/GCM_BA/                           (EOS; inputs read directly from GCM_downscaling/Input_data/ and EC-Earth_hosing/)
├── 2_F1_calculation/    F1_<period>/       : WDF rule + QDM parameters
├── GCM_bias_adjusted_0.7deg/  <run>_F1_<period>.nc : historical + 4 GCM runs bias-adjusted (ERA5-like), EC-Earth3 grid
└── 4_verification/cache/                   : statistics of the verification
```

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE                # rsds not bias-adjusted (to add it back: VARS_CORE + cfg.get("extra_variables", []))
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

GRID_FROM = "gcm_historical"   # the EC-Earth3 'gr' grid (historical run), shared by the GCM runs
print("variables:", VARS_ALL)


## 1. Every input on the EC-Earth3 grid: period, number of days

In [ ]:
GRID = bu.read_raw(cfg, GRID_FROM, "tasmax").isel(time=0, drop=True)
rows = []
for name in ["era5", "gcm_historical"] + GCM_RUNS:
    for v in VARS_ALL:
        if not bu.gcm_ba_available(cfg, name, v):
            rows.append({"input": name, "var": v, "status": "missing (config, notebook 00)"}); continue
        da = bu.gcm_ba_input(cfg, name, v)
        yrs = np.unique(da.time.dt.year.values)
        rows.append({"input": name, "var": v, "status": "ok", "same grid": bu.same_grid(da.isel(time=0, drop=True), GRID),
                     "period": f"{yrs[0]}-{yrs[-1]}", "n days": da.sizes["time"], "expected (noleap)": 365 * len(yrs),
                     "units": da.attrs.get("units"), "orig. units": da.attrs.get("orig_units"),
                     "orig. calendar": da.attrs.get("orig_calendar")})
inputs = pd.DataFrame(rows).set_index(["input", "var"])
display(inputs)
ok = inputs[inputs["status"] == "ok"]
assert ok["same grid"].all(), "some inputs are not on the EC-Earth3 grid"
cal = ok.loc[["era5", "gcm_historical"]]
assert (cal["n days"] == cal["expected (noleap)"]).all(), "ERA5 / EC-Earth3 historical: missing or extra days"
part = ok[ok["n days"] != ok["expected (noleap)"]]
if len(part):
    print("[!] incomplete years (to check):", list(part.index))
assert (cal["period"].str[:4] <= CAL_PERIOD[0]).all() and (cal["period"].str[-4:] >= CAL_PERIOD[1]).all(), \
    f"ERA5 / EC-Earth3 historical do not cover {CAL_PERIOD}"
print(f"ok: every input on the EC-Earth3 grid ({GRID.sizes['lat']} x {GRID.sizes['lon']}), complete years, calibration period {CAL_PERIOD} covered")

## 2. Regridding: CDO against Python

ERA5 0.25 deg, ERA5 regridded by CDO and by Python (`ba_utils.regrid_conservative`) over the first 30 days: global means
(conservation) and CDO - Python differences. Both are conservative: they should agree closely (small differences: CDO
uses the exact cell bounds of the Gaussian grid, Python the mid-points between centres). Then one day on a map.

In [ ]:
rows = []
for v in VARS_ALL:
    if not (bu.dataset_available(cfg, "era5", v) and bu.gcm_ba_available(cfg, "era5", v)):
        continue
    e = bu.read_raw(cfg, "era5", v).isel(time=slice(0, 30))
    p = bu.regrid_conservative(e, GRID.lat.values, GRID.lon.values).compute()
    c = bu.gcm_ba_input(cfg, "era5", v).sel(time=e.time).compute()
    d = np.abs(c - p.assign_coords(lat=c.lat, lon=c.lon))
    a = float(bu.area_mean(e).mean())
    rows.append({"var": v, "ERA5 0.25 deg": a, "CDO": float(bu.area_mean(c).mean()), "Python": float(bu.area_mean(p).mean()),
                 "CDO - ERA5 (%)": 100 * (float(bu.area_mean(c).mean()) - a) / (abs(a) + 1e-9),
                 "max |CDO - Python|": float(d.max()), "mean |CDO - Python|": float(bu.area_mean(d).mean())})
display(pd.DataFrame(rows).set_index("var").round(4))

In [ ]:
v = "tasmax"
e1 = bu.read_raw(cfg, "era5", v).isel(time=slice(0, 1))
e = e1.isel(time=0).compute()
c = bu.gcm_ba_input(cfg, "era5", v).isel(time=0).compute()
p = bu.regrid_conservative(e1, GRID.lat.values, GRID.lon.values).isel(time=0).compute()
dd = c - p.assign_coords(lat=c.lat, lon=c.lon)
lim = float(np.nanpercentile(np.abs(dd), 99.9)) or 1.0
panels = [(e, f"ERA5 0.25 deg: {v}", {}), (c, f"ERA5 on the EC-Earth3 grid (CDO): {v}", {}),
          (dd, f"CDO - Python: {v}", dict(cmap="RdBu_r", vmin=-lim, vmax=lim))]
fig, axes = plt.subplots(1, 3, figsize=(24, 4))
for ax, (d, t, kw) in zip(axes, panels):
    im = ax.pcolormesh(d.lon, d.lat, d, shading="auto", **kw); plt.colorbar(im, ax=ax, shrink=0.8); ax.set_title(t)
    ax.set_facecolor("0.85")
plt.tight_layout(); plt.show()